In [ ]:
# Vulnerability CLassifier (NN): i feel like I would use a normal ANN
# GOAL: Given vulnerability attributes and description predict a CVSS score (float value from 0-10)
# INPUT:
# OUTPUT: value between 0-10
# https://www.kaggle.com/code/cloudnineforreal/cvss-prediction

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
# import torch
# import torch.nn as nn
# import torch.optim as optim

## Upload/Understand Data

In [ ]:
cve_data = pd.read_csv("../data/cve.csv")
cve_data.head()

In [ ]:
# drop columns
drop_col = ["Unnamed: 0", "mod_date", "pub_date"]
cve_data.drop(columns=drop_col, inplace=True)

# fill na categorical columns as "UNKNOWN"
catgy_cols = ["access_authentication", "access_complexity", "access_vector", "impact_availability", "impact_confidentiality", "impact_integrity"]
cve_data[catgy_cols] = cve_data[catgy_cols].fillna("UNKNOWN")

# one hot encode categorical columns
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).set_output(transform="pandas")
catgy_encode = ohe.fit_transform(cve_data[catgy_cols])

# combine data
cve_data = pd.concat([cve_data, catgy_encode], axis=1)
cve_data.drop(columns=catgy_cols, inplace=True)

In [ ]:
cve_data["summary"]

In [ ]:
# vectorize summary field
model = SentenceTransformer("all-MiniLM-L6-v2")  # TRY TO USE SBERT
embeddings = model.encode(cve_data["summary"])
embeddings

# tfidf_summary = TfidfVectorizer(max_features=500, stop_words="english")
# summary_feat = tfidf_summary.fit_transform(cve_data["summary"])
# # print(summary_feat[6])
# summary_feat_df = pd.DataFrame(
#     summary_feat.toarray(),
#     columns=[f"tfidf_summary_{i}" for i in range(summary_feat.shape[1])]
# )

# # combine data
# merged_cve_data = pd.concat([cve_data.drop(columns=["summary"]), summary_feat_df], axis=1)

# # vectorize cve name field
# tfidf_name = TfidfVectorizer(max_features=50, stop_words="english")
# cwe_name_feat = tfidf_name.fit_transform(cve_data["cwe_name"])
# name_feat_df = pd.DataFrame(
#     cwe_name_feat.toarray(),
#     columns=[f"tfidf_name_{i}" for i in range(cwe_name_feat.shape[1])]
# )

# # combine data
# merged_cve_data = pd.concat([cve_data.drop(columns=["cwe_name"]), name_feat_df], axis=1)
# merged_cve_data.head()

In [ ]:
# split train/test
input_cols = merged_cve_data.loc[:, merged_cve_data.columns != "cvss"].columns
# input_cols
X = merged_cve_data[input_cols]
y = merged_cve_data["cvss"]

# drop object columns
X = X.select_dtypes(exclude="object")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## Model Definition

In [ ]:
rfr = RandomForestRegressor(n_estimators=10)
rfr.fit(X_train, y_train)

In [ ]:
# model metrics
from sklearn.metrics import mean_squared_error, r2_score

# oob = rfr.oob_score_
# print(f"Out of Bag Score: {oob}")

predict = rfr.predict(X_test)
mse = mean_squared_error(y_test, predict)
print(f"MSE: {mse}")

r2 = r2_score(y_test, predict)
print(f"R2 Value: {r2}")

"""TFIDF METRICS
MSE: 0.02974535224109783
R2 Value: 0.9925021166467017
"""


In [ ]:
# visualize RFR
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

tree_to_plot = rfr.estimators_[0]
plt.figure(figsize=(20,10))
plot_tree(tree_to_plot, feature_names=merged_cve_data.columns.tolist(), filled=True, rounded=True, fontsize=10)
plt.show()